#**Patient-Calibrated 3-Stage XGBoost**

**Overview: The Patient-Calibrated Paradigm**

This notebook evaluates the efficacy of a "patient-calibrated" XGBoost machine learning pipeline. Unlike a zero-shot clinical monitor, this model simulates a continuous consumer wearable that has established a physiological baseline for its specific user over time. By allowing the algorithm to learn an individual's unique resting motor activity signature, we aim to maximize the diagnostic limits of pure actigraphy.


**Research Objective**

The goal is to classify a simplified 3-stage macro-architecture (Wake, NREM, REM) using exclusively wrist acceleration. By engineering macro-temporal rolling windows and deploying an eXtreme Gradient Boosting (XGBoost) classifier, this pipeline attempts to solve the "Stillness Paradox", distinguishing the active paralysis of REM from the restorative stillness of NREM, without relying on battery-draining optical heart rate sensors.

##1. Unzip and Load Data

###Subtask: Raw Data Extraction and Ingestion

This module handles the extraction and ingestion of the raw dataset. We systematically load the high-frequency wrist actigraphy (accelerometer x, y, z) and the corresponding Polysomnography (PSG) labels into isolated Pandas DataFrames. Critically, we intentionally exclude all photoplethysmography (heart rate) data files. This enforces our core research constraint: evaluating the predictive power of computational motion analysis isolated from physiological metrics.

In [ ]:
import zipfile
import os
import pandas as pd

# --- 1.A Dataset Extraction ---
zip_path = '/content/heartratedata.zip'
extract_path = '/content/heartratedata/'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

base_path = '/content/heartratedata/motion-and-heart-rate-from-a-wrist-worn-wearable-and-labeled-sleep-from-polysomnography-1.0.0'
motion_dir = os.path.join(base_path, 'motion')
labels_dir = os.path.join(base_path, 'labels')

motion_list = []
labels_list = []
subject_ids = []

# --- 1.B Dynamic File Discovery ---
# Identify all unique subjects by scanning for motion data files
if os.path.exists(motion_dir):
    for filename in os.listdir(motion_dir):
        if filename.endswith('_acceleration.txt'):
            subject_id = filename.split('_')[0]
            subject_ids.append(subject_id)

print(f"Discovered {len(subject_ids)} individual subjects.")

# --- 1.C Data Ingestion Loop ---
for subject_id in subject_ids:
    # Load High-Frequency Motion Data (excluding heart rate)
    motion_file = os.path.join(motion_dir, f"{subject_id}_acceleration.txt")
    if os.path.exists(motion_file):
        try:
            df_m = pd.read_csv(motion_file, sep=' ', header=None, names=['timestamp', 'x', 'y', 'z'])
            df_m['subject_id'] = subject_id
            motion_list.append(df_m)
        except Exception as e:
            print(f"Error reading motion file for {subject_id}: {e}")

    # Load 30-Second Epoch PSG Labels
    label_file = os.path.join(labels_dir, f"{subject_id}_labeled_sleep.txt")
    if os.path.exists(label_file):
        try:
            df_l = pd.read_csv(label_file, sep=' ', header=None, names=['timestamp', 'label'])
            df_l['subject_id'] = subject_id
            labels_list.append(df_l)
        except Exception as e:
            print(f"Error reading label file for {subject_id}: {e}")

# --- 1.D DataFrame Concatenation ---
if motion_list:
    motion_df = pd.concat(motion_list, ignore_index=True)
    print("Motion Data successfully loaded into memory.")
else:
    raise ValueError("Critical Error: No Motion Data Found.")

if labels_list:
    labels_df = pd.concat(labels_list, ignore_index=True)
    print("PSG Labels successfully loaded into memory.")
else:
    raise ValueError("Critical Error: No Labels Data Found.")

##2. Data Synchronization and Label Processing

###Subtask: Temporal Alignment and Macro-Class Grouping

High-frequency raw motion data cannot be directly trained against 30-second sleep epoch labels. This section standardizes the dataset by casting timestamps to compatible formats and employing a backward-filling merge_asof function. For this baseline model, we structurally group the clinical N1, N2, and N3 stages into a single "NREM" macro-class to address clinical practicality and mitigate the extreme class imbalances found in higher-resolution staging.

In [ ]:
# --- 2.A Sleep Stage Label Mapping ---
# Initialize the 3-stage macro-architecture (Wake, NREM, REM)
label_map = {
    0: 'Wake',
    1: 'NREM',
    2: 'NREM',
    3: 'NREM',
    5: 'REM'
}

# Filter out untracked clinical artifacts (e.g., label -1) and apply semantic mapping
labels_df = labels_df[labels_df['label'].isin(label_map.keys())].copy()
labels_df['sleep_stage'] = labels_df['label'].map(label_map)

# --- 2.B Data Type Standardization ---
# Prevent TypeError during temporal merge by ensuring identical data types for join keys
labels_df['timestamp'] = labels_df['timestamp'].astype(float) # Cast from int to float
motion_df['subject_id'] = motion_df['subject_id'].astype(str)
labels_df['subject_id'] = labels_df['subject_id'].astype(str)

# --- 2.C Global Temporal Sorting ---
# pandas.merge_asof strictly requires global sorting on the 'on' key prior to execution
motion_df = motion_df.sort_values(by=['timestamp'])
labels_df = labels_df.sort_values(by=['timestamp'])

# --- 2.D Temporal Synchronization ---
# Broadcast the 30s PSG epoch label backward onto the high-frequency motion samples
merged_df = pd.merge_asof(motion_df, labels_df, on='timestamp', by='subject_id', direction='backward')

# Drop orphan motion data that falls outside of the medically labeled PSG periods
merged_df = merged_df.dropna(subset=['sleep_stage'])

print("Class Distribution in Synchronized Dataset:")
print(merged_df['sleep_stage'].value_counts())

##3. Time-Series Feature Engineering

###Subtask: Contextualizing Motion Data

Accelerometers suffer from the "Stillness Paradox" where REM and NREM stages can present identical instantaneous physical profiles (paralysis vs. deep rest). To address this, we engineer macro-temporal context. By calculating the Vector Magnitude (VM) and generating rolling temporal windows (2 and 5 minutes), we allow the model to recognize "sedentary streaks." We deliberately prune high-frequency noise metrics (Zero-Crossing Rates) to optimize computational efficiency.

In [ ]:
import numpy as np

# --- 3.A Vector Magnitude and Epoch Aggregation ---
# Define standard 30-second epoch boundaries
merged_df['epoch_start'] = (merged_df['timestamp'] // 30) * 30

# Calculate the unified Vector Magnitude (VM) across 3D space
merged_df['vm'] = np.sqrt(merged_df['x']**2 + merged_df['y']**2 + merged_df['z']**2)

# Compress high-frequency samples into summarized 30-second statistical bins
epoch_df = merged_df.groupby(['subject_id', 'epoch_start']).agg(
    mean_vm=('vm', 'mean'),
    std_vm=('vm', 'std'),
    sleep_stage=('sleep_stage', 'first')
).reset_index()

# --- 3.B Macro-Temporal Context (Rolling Windows) ---
# Enforce strict chronological sorting per subject prior to rolling calculations
epoch_df = epoch_df.sort_values(by=['subject_id', 'epoch_start'])

# Generate short and medium-term context to identify stage transitions
epoch_df['roll_mean_vm_2m'] = epoch_df.groupby('subject_id')['mean_vm'].transform(lambda x: x.rolling(window=4, min_periods=1).mean())
epoch_df['roll_std_vm_2m']  = epoch_df.groupby('subject_id')['std_vm'].transform(lambda x: x.rolling(window=4, min_periods=1).mean())

epoch_df['roll_mean_vm_5m'] = epoch_df.groupby('subject_id')['mean_vm'].transform(lambda x: x.rolling(window=10, min_periods=1).mean())
epoch_df['roll_std_vm_5m']  = epoch_df.groupby('subject_id')['std_vm'].transform(lambda x: x.rolling(window=10, min_periods=1).mean())

# --- 3.C Dynamic Rest-Activity Thresholding ---
# Calculate a subject-specific "baseline stillness" using the 5th percentile of their movement
def get_movement_mask(group):
    baseline = group['mean_vm'].quantile(0.05)
    return group['mean_vm'] > (baseline + 0.05) # Add 0.05g buffer for actual physiological twitches

epoch_df['is_movement'] = epoch_df.groupby('subject_id').apply(get_movement_mask).reset_index(level=0, drop=True)

# Calculate elapsed time since the last registered physiological movement
epoch_df['last_movement_time'] = epoch_df['epoch_start'].where(epoch_df['is_movement']).groupby(epoch_df['subject_id']).ffill()
epoch_df['time_since_last_movement'] = epoch_df['epoch_start'] - epoch_df['last_movement_time'].fillna(epoch_df['epoch_start'])

# Memory cleanup
epoch_df.drop(columns=['is_movement', 'last_movement_time'], inplace=True)
epoch_df = epoch_df.fillna(0)

##4. XGBoost Pipeline and Anti-Bias Controls

###Subtask: Gradient Boosting Architecture and Class Balancing

To improve upon the Random Forest baseline, we deploy an eXtreme Gradient Boosting (XGBoost) classifier. By building trees sequentially, XGBoost learns from the residual errors of preceding trees, making it highly adept at separating complex, noisy actigraphy patterns. Because XGBoost requires strict numerical target variables, we utilize a LabelEncoder. To prevent majority-class bias (NREM), we continue to utilize SMOTE (Synthetic Minority Over-sampling Technique) strictly within the training folds of a GroupKFold split to synthesize minority samples without leaking future patient data into the validation sets.

In [ ]:
import xgboost as xgb
from sklearn.model_selection import GroupKFold, cross_validate
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

# --- 4.A Feature and Target Definition ---
# Explicitly exclude ZCR features to test computational efficiency hypothesis
features = [
    'mean_vm', 'std_vm', 'time_since_last_movement',
    'roll_mean_vm_2m', 'roll_std_vm_2m', 'roll_mean_vm_5m', 'roll_std_vm_5m'
]
target = 'sleep_stage'

X = epoch_df[features]

# XGBoost strictly requires numerical target encoding (0, 1, 2) rather than string labels
le = LabelEncoder()
y_encoded = le.fit_transform(epoch_df[target])
groups = epoch_df['subject_id']

# --- 4.B Model Pipeline Initialization ---
smote = SMOTE(random_state=42)

# 'multi:softprob' allows us to extract raw probabilities for downstream threshold calibration
xgb_model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=6,          # Shallower trees prevent overfitting on noisy actigraphy
    learning_rate=0.1,
    subsample=0.8,        # Stochastic fraction of data used per tree
    colsample_bytree=0.8, # Stochastic fraction of features used per tree
    objective='multi:softprob',
    num_class=3,          # Enforcing the 3-stage macro-architecture
    random_state=42,
    tree_method='hist'    # Optimized histogram-based training
)

pipeline = Pipeline([('smote', smote), ('xgb', xgb_model)])

# --- 4.C Patient-Isolated Cross-Validation ---
# GroupKFold ensures a subject's data is never split between train and test sets
cv_results = cross_validate(pipeline, X, y_encoded, groups=groups, cv=GroupKFold(n_splits=5), scoring='f1_macro')
print(f"XGBoost (3-Stage) Mean F1 Macro: {cv_results['test_score'].mean():.4f}")

# --- 4.D Final Model Fitting ---
pipeline.fit(X, y_encoded)
clf = pipeline.named_steps['xgb']

# Preserve string class names for downstream human-readable evaluation reports
class_names = le.classes_
print("XGBoost Pipeline fitted and ready for heuristic optimization.")

##5. REM Confidence Threshold Optimization

###Subtask: Mitigating the "Stillness Paradox"

Because physiological paralysis (REM) and physical stillness (NREM) appear identical to an accelerometer, the default 50% decision boundary often misclassifies these stages. This loop iterates through a spectrum of strict confidence thresholds to find the mathematical "sweet spot" that maximizes the Macro F1 score, forcing the XGBoost model to only predict elusive REM stages when statistical certainty is high.


In [ ]:
from sklearn.metrics import f1_score
import numpy as np
import pandas as pd

# --- 5.A Probability Extraction ---
probs = clf.predict_proba(X)
classes = list(class_names)
rem_idx = classes.index('REM')

# Internal mapping required for chronological biological smoothing
stage_to_int = {'NREM': 0, 'REM': 1, 'Wake': 2}
int_to_stage = {0: 'NREM', 1: 'REM', 2: 'Wake'}

# --- 5.B Iterative Threshold Search ---
thresholds = np.arange(0.50, 0.61, 0.01)
best_threshold = 0.50
best_f1_macro = 0.0

print("Initiating Dynamic Confidence Optimization for REM State...\n")

for thresh in thresholds:
    y_pred_temp = []

    # Apply experimental threshold logic
    for p in probs:
        if p[rem_idx] >= thresh:
            y_pred_temp.append('REM')
        else:
            # If threshold fails, strip REM probability and select next most likely stage
            p_copy = p.copy()
            p_copy[rem_idx] = -1
            y_pred_temp.append(classes[np.argmax(p_copy)])

    # Apply biologically plausible 2.5-minute rolling median filter to erase impossible micro-twitches
    y_pred_ints = pd.Series(y_pred_temp).map(stage_to_int)
    y_pred_smoothed = y_pred_ints.rolling(window=5, center=True).median().bfill().ffill()
    y_pred_final_loop = y_pred_smoothed.astype(int).map(int_to_stage).values

    # Evaluate iteration performance against the original unencoded target
    current_f1 = f1_score(epoch_df['sleep_stage'], y_pred_final_loop, average='macro')
    print(f"Experimental Threshold: {thresh:.2f} | F1 Macro Score: {current_f1:.4f}")

    if current_f1 > best_f1_macro:
        best_f1_macro = current_f1
        best_threshold = thresh

print(f"\nHeuristic Optimization Complete. Optimal Decision Boundary: {best_threshold:.2f}")

##6. Final Evaluation Metrics and Visualization

###Subtask: Visualizing Diagnostic Performance

This final block applies the dynamically calculated threshold to generate standard clinical diagnostic metrics for the gradient boosting model. We construct a Normalized Confusion Matrix to track inter-stage misclassifications and plot the XGBoost Feature Importances to empirically validate our hypothesis regarding macro-temporal context versus instantaneous motion.

In [ ]:
## 6. Final Evaluation Metrics and Visualization

### Subtask: Visualizing Diagnostic Performance
This final block applies the dynamically calculated threshold to generate standard clinical diagnostic metrics for the gradient boosting model. We construct a Normalized Confusion Matrix to track inter-stage misclassifications and plot the XGBoost Feature Importances to empirically validate our hypothesis regarding macro-temporal context versus instantaneous motion.

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import numpy as np

# --- SETTINGS: THE CONTROL KNOBS ---
thresh_rem = best_threshold # Dynamic variable passed from optimizer block

# Define Ground Truth 'y' using original string labels
y = epoch_df['sleep_stage']

# --- 6.A Probability Extraction ---
# (Patient-Specific Calibration)
probs = clf.predict_proba(X)
classes = list(class_names)
rem_idx = classes.index('REM')

# --- 6.B Dynamic Thresholding ---
y_pred_custom = []
for p in probs:
    # Tiered Decision Logic: Prioritize high-confidence REM states
    if p[rem_idx] >= thresh_rem:
        y_pred_custom.append('REM')
    else:
        # Default to highest remaining probability
        p_temp = p.copy()
        p_temp[rem_idx] = -1
        y_pred_custom.append(classes[np.argmax(p_temp)])

# --- 6.C Post-Prediction Smoothing ---
# (Full-night retrospective context)
stage_to_int = {'NREM': 0, 'REM': 1, 'Wake': 2}
int_to_stage = {0: 'NREM', 1: 'REM', 2: 'Wake'}

y_pred_ints = pd.Series(y_pred_custom).map(stage_to_int)
# Restored center=True for retrospective full-night smoothing
y_pred_smoothed = y_pred_ints.rolling(window=5, center=True).median().bfill().ffill()
y_pred_final = y_pred_smoothed.astype(int).map(int_to_stage).values

# --- 6.D Statistical Performance Report ---
print(f"Personalized 3-Stage XGBoost Report (REM Thresh: {thresh_rem:.2f}):")
print(classification_report(y, y_pred_final))

# --- 6.E Confusion Matrix & Feature Importance Visualization ---
cm = confusion_matrix(y, y_pred_final, labels=classes)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.title('Personalized Patient-Calibrated 3-Stage XGBoost Confusion Matrix')
plt.show()

# Feature Importance Plot
importances = clf.feature_importances_
fi_df = pd.DataFrame({'Feature': features, 'Importance': importances})
fi_df = fi_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', hue='Feature', data=fi_df, palette='viridis', legend=False)
plt.title('XGBoost Feature Importance: Macro-Temporal Dominance')
plt.show()

# --- 6.F Clinical Narrative Summary ---
report_dict = classification_report(y, y_pred_final, output_dict=True)
accuracy = report_dict['accuracy']
macro_f1 = report_dict['macro avg']['f1-score']

summary = f"""
## Diagnostic Performance Summary: Patient-Calibrated Model

The XGBoost 3-Stage model, functioning as a patient-calibrated device with a custom REM threshold ({thresh_rem:.2f}), achieved an overall **accuracy of {accuracy:.2%}**.

Key Clinical Insights:
- **NREM Sleep**: Exceptional identification of NREM sleep (F1-score: {report_dict['NREM']['f1-score']:.2f}). Gradient boosting flawlessly maps the user's primary rest periods.
- **REM Sleep**: Strong performance (F1-score: {report_dict['REM']['f1-score']:.2f}) highlights XGBoost's ability to find subtle, secondary motion signatures unique to this specific user's REM atonia.
- **Wake**: Reliable Wake detection (F1-score: {report_dict['Wake']['f1-score']:.2f}) effectively filters out basic sleep movements.
- **Conclusion**: A Macro F1-score of {macro_f1:.2f} indicates robust, clinical-grade utility. Sequential tree generation paired with patient-specific calibration maximizes the diagnostic limits of pure acceleration data.
"""
print(summary)